# Video Analysis: Shots, Scenes and Face Groups

This notebook analyses one YouTube video and produces the assignment deliverables:

| Output | How it is produced |
|---|---|
| **Shots** | Camera cuts detected locally with PySceneDetect |
| **Scenes** with descriptions | Consecutive shots grouped and described by a vision-language model on OpenRouter, using sampled frames and a local Whisper transcript |
| **Face groups** | Faces detected locally with InsightFace and clustered by embedding similarity |

Results are written to `outputs/analysis.json` and `outputs/extracted_faces/scene_NNN/group_NNN/`. The design is explained in `README.md`.

> Face groups are anonymous similarity clusters, not verified identities, and scene boundaries and descriptions are model output. Review both before relying on them.

## 1. Install dependencies

Run once per environment (Python 3.11 or newer), then restart the kernel. The first pipeline run also downloads the InsightFace and Whisper model weights.

In [ ]:
from pathlib import Path

ROOT = Path.cwd() if (Path.cwd() / "pyproject.toml").exists() else Path.cwd().parent
%pip install -q -e "{ROOT}"

## 2. Configure

Secrets come from a `.env` file in the project root (copy `.env.example`): `OPENROUTER_API_KEY` and, optionally, `OPENROUTER_MODEL`. Place the downloaded video at `VIDEO_PATH` and set its YouTube link below.

In [ ]:
import json
import os
import sys
from collections import defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from dotenv import load_dotenv

from video_analysis.evaluation import boundary_metrics, grouping_metrics
from video_analysis.faces import FaceGrouper
from video_analysis.media import probe
from video_analysis.pipeline import run_analysis
from video_analysis.scenes import SemanticAnalyzer
from video_analysis.schema import strict_json_loads

ROOT = Path.cwd() if (Path.cwd() / "pyproject.toml").exists() else Path.cwd().parent
load_dotenv(ROOT / ".env", override=True)

VIDEO_PATH = ROOT / "data" / "input" / "episode.mp4"
YOUTUBE_URL = "https://www.youtube.com/watch?v=7uM4rcXXyiM&list=PLWCymGfobuvg"
OUTPUT_DIR = ROOT / "outputs"
CACHE_DIR = ROOT / "cache"

OPENROUTER_MODEL = os.environ.get("OPENROUTER_MODEL", "openrouter/free")
SAMPLE_FPS = 2.0         # frames sampled per second of each shot
WHISPER_MODEL = "small"  # faster-whisper model size
FACE_SIMILARITY = 0.48   # cosine similarity a face needs to join an existing group
MAX_API_REQUESTS = 100   # hard cap on OpenRouter calls, retries included

print(f"Python {sys.version.split()[0]} | {VIDEO_PATH.name} | {OPENROUTER_MODEL}")

## 3. Inspect the input

In [ ]:
video = probe(VIDEO_PATH)
print(f"{video.filename}: {video.duration_s / 60:.1f} min, {video.width}x{video.height}, "
      f"sha256 {video.sha256[:12]}")

## 4. Load the face model and check the LLM

The preflight sends a tiny green image to the configured OpenRouter model and expects `{"color": "green"}` back as strict JSON, so a model without image or structured-output support fails here in seconds instead of halfway through the run.

In [ ]:
face_grouper = FaceGrouper(similarity_threshold=FACE_SIMILARITY)
face_grouper.load_model()

semantic_analyzer = SemanticAnalyzer(OPENROUTER_MODEL, CACHE_DIR / "semantic", max_requests=MAX_API_REQUESTS)
semantic_analyzer.preflight()
print("Face model loaded; the LLM accepts images and returns strict JSON.")

## 5. Run the pipeline

```mermaid
flowchart LR
  V[Video file] --> S[Shot detection]
  V --> T[Whisper transcript]
  S --> F[Frame sampling]
  F --> G[Face detection and grouping]
  S & T & F & G --> B[LLM: scene boundaries]
  B --> D[LLM: scene descriptions]
  D --> C[Validation]
  C --> O[analysis.json and extracted_faces/]
```

Each stage is cached in `cache/`, keyed by the video's hash and the settings it depends on, and every accepted LLM answer is cached too. Re-running is therefore cheap, and a run interrupted by a rate limit resumes where it stopped. Every model call is logged, without content or keys, to `cache/semantic/llm-traces.jsonl`.

In [ ]:
analysis = run_analysis(VIDEO_PATH, YOUTUBE_URL, OUTPUT_DIR, CACHE_DIR, face_grouper, semantic_analyzer,
                        sample_fps=SAMPLE_FPS, whisper_model=WHISPER_MODEL)

group_count = len({face.group_id for face in analysis.faces})
print(f"{len(analysis.shots)} shots, {len(analysis.scenes)} scenes, {len(analysis.transcript)} transcript segments, "
      f"{len(analysis.faces)} faces in {group_count} groups")
print(f"OpenRouter requests: {semantic_analyzer.requests_made}, cached answers reused: {semantic_analyzer.cache_hits}")

## 6. Review the scenes

`review` marks scenes where the model reported uncertainty about the description or the boundary that opens the scene.

In [ ]:
scene_table = pd.DataFrame([{
    "scene": number, "start_s": round(scene.start_s, 1), "end_s": round(scene.end_s, 1),
    "shots": len(scene.shot_ids), "kind": scene.kind, "face_groups": ", ".join(scene.face_group_ids),
    "review": scene.needs_review, "description": scene.description,
} for number, scene in enumerate(analysis.scenes)])
with pd.option_context("display.max_colwidth", None):
    display(scene_table)

fig, ax = plt.subplots(figsize=(12, 1.8))
for number, scene in enumerate(analysis.scenes):
    ax.barh(0, scene.end_s - scene.start_s, left=scene.start_s, height=0.5,
            color=plt.cm.tab20(number % 20), edgecolor="white")
    ax.text((scene.start_s + scene.end_s) / 2, 0, str(number), ha="center", va="center", fontsize=8)
for shot in analysis.shots[1:]:
    ax.vlines(shot.start_s, 0.4, 0.8, color="grey", linewidth=0.6)
ax.set(xlabel="Time (s)", ylim=(-0.5, 1), yticks=[], title="Scenes (numbered bars) and shot cuts (grey ticks)")
plt.show()

## 7. Review the face groups

Each row shows crops spread across one group, largest groups first. A row that mixes different people means the similarity threshold is too low; one person spread over several rows means it is too high.

In [ ]:
CROPS_PER_GROUP, MAX_GROUPS_SHOWN = 8, 15

crops_by_group = defaultdict(list)
for path in sorted((OUTPUT_DIR / "extracted_faces").glob("scene_*/group_*/*.jpg")):
    crops_by_group[path.parent.name].append(path)
groups = sorted(crops_by_group, key=lambda group: -len(crops_by_group[group]))[:MAX_GROUPS_SHOWN]

fig, axes = plt.subplots(len(groups), CROPS_PER_GROUP, squeeze=False,
                         figsize=(CROPS_PER_GROUP * 1.2, len(groups) * 1.4))
for row, group in zip(axes, groups):
    paths = crops_by_group[group]
    count = min(CROPS_PER_GROUP, len(paths))
    for axis in row:
        axis.axis("off")
    for axis, index in zip(row, range(count)):
        axis.imshow(plt.imread(paths[index * len(paths) // count]))
    row[0].set_title(f"{group}: {len(paths)} faces", fontsize=8, loc="left")
plt.tight_layout()
plt.show()

## 8. Optional: measure accuracy

The pipeline's automatic checks prove the output is internally consistent, not that it is correct. To measure accuracy, label a sample by hand below and re-run the cell. Face crops are labelled by their path under `outputs/extracted_faces/`, which already contains the predicted group.

In [ ]:
TRUE_CUTS_S = []          # times (s) of real camera cuts
TRUE_SCENE_STARTS_S = []  # times (s) where real scenes start, excluding 0
TRUE_PEOPLE = {}          # crop path -> person, e.g. {"scene_001/group_002/face_000_00-00-01.960.jpg": "woman_on_couch"}

if TRUE_CUTS_S:
    print("Shot cuts:", boundary_metrics([shot.start_s for shot in analysis.shots[1:]], TRUE_CUTS_S, tolerance_s=0.1))
if TRUE_SCENE_STARTS_S:
    print("Scene starts:", boundary_metrics([scene.start_s for scene in analysis.scenes[1:]],
                                              TRUE_SCENE_STARTS_S, tolerance_s=1.0))
if TRUE_PEOPLE:
    print("Face groups:", grouping_metrics(list(TRUE_PEOPLE.values()),
                                           [Path(path).parent.name for path in TRUE_PEOPLE]))
if not (TRUE_CUTS_S or TRUE_SCENE_STARTS_S or TRUE_PEOPLE):
    print("No manual labels provided; accuracy was not measured.")

## 9. Verify the submission files

Re-reads `analysis.json` from disk, checks it has exactly the required fields, and checks that every face group listed for a scene has its folder of crops.

In [ ]:
submission = strict_json_loads((OUTPUT_DIR / "analysis.json").read_text())
assert list(submission) == ["video_url", "shots", "scenes"]
assert all(list(shot) == ["id", "start_seconds", "end_seconds"] for shot in submission["shots"])
assert all(list(scene) == ["id", "start_seconds", "end_seconds", "description", "faces"]
           for scene in submission["scenes"])
for scene in submission["scenes"]:
    for group_id, timecodes in scene["faces"].items():
        folder = OUTPUT_DIR / "extracted_faces" / f"scene_{scene['id']:03d}" / group_id
        assert len(list(folder.glob("*.jpg"))) >= len(timecodes), f"Missing crops in {folder}"

print(f"analysis.json is valid: {len(submission['shots'])} shots, {len(submission['scenes'])} scenes")
print(json.dumps(submission["scenes"][0], indent=2))

## Submission

- **Code**: `src/video_analysis/`, this notebook and `tests/`
- **Video link**: `YOUTUBE_URL` above, also stored in `analysis.json`
- **JSON**: `outputs/analysis.json`
- **Face folders**: `outputs/extracted_faces/scene_NNN/group_NNN/`

**Known limitations.** Face groups can split one person (profile vs. frontal, lighting) or merge look-alikes; faces are only found in sampled frames, so very brief appearances can be missed; scene boundaries and descriptions come from a model and can be wrong, especially for flashbacks and montages. The free OpenRouter route may pick a different underlying model per request, so descriptions can vary in style.